# Reproducibility and Inspection

Run yesterday's experiment again and the loss curve comes out different.
Is this morning's change an improvement, or a lucky seed? Answering that
question requires knowing where every random number in a training run comes
from. And once two runs *do* disagree, or a loss turns into NaN, the next
question is what the model computed layer by layer, ideally without editing
its source. This section covers both skills: controlling randomness (seeds,
generators, determinism) and observing a running model from the outside
(hooks).

In [ ]:
import jax
from jax import numpy as jnp
from flax import nnx
import optax

## Seeds and Randomness

Randomness enters a training run in more places than you might list on a
first try: initialization draws every weight (that section),
dropout samples a fresh mask at each step
[@Srivastava.Hinton.Krizhevsky.ea.2014], the data loader shuffles
examples differently in every epoch, augmentations sample crops and flips,
and the loader's worker processes each do all of this in parallel. Every one
of these consults a pseudorandom number generator, a deterministic algorithm
whose entire output sequence is fixed by its seed. Seed every generator and
the run is repeatable; miss one and it is not.

JAX has no global generator to seed. Every random operation takes a *key*,
a value that fully determines its output: `jax.random.normal(key, shape)`
returns the same numbers every time it is called with that key, and fresh
randomness comes only from deriving fresh keys with `jax.random.split`. The
function below turns its seed into one key and splits it three ways, one
key each for initialization, inputs, and targets, so the seed pins the
whole run by construction:

In [ ]:
def train_once(seed):
    key_init, key_X, key_y = jax.random.split(jax.random.key(seed), 3)
    rngs = nnx.Rngs(key_init)
    net = nnx.Sequential(nnx.Linear(20, 32, rngs=rngs),
                         nnx.relu,
                         nnx.Linear(32, 1, rngs=rngs))
    X = jax.random.normal(key_X, (128, 20))
    y = jax.random.normal(key_y, (128, 1))
    optimizer = nnx.Optimizer(net, optax.sgd(0.1), wrt=nnx.Param)
    loss_fn = lambda model: ((model(X) - y) ** 2).mean()
    for _ in range(5):
        loss, grads = nnx.value_and_grad(loss_fn)(net)
        optimizer.update(net, grads)
    return loss, net.layers[0].kernel[...]

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, jnp.array_equal(w_a, w_b))
print('different seed:', loss_a, 'vs', loss_c)

The two seeded runs agree *bitwise*, down to the last floating-point bit:
same initial weights, same data, same gradients, same operations in the
same order. One detail deserves a flag: we pinned the TensorFlow run to
the CPU. On a GPU the same seeded code produces equal losses but weights
that differ in their last bits — the reason emerges in the next section.

### Generator Objects

A single shared stream is fragile: every consumer advances it, so
inserting one extra random call shifts everything drawn after it. In JAX
the keys *are* the generator objects: each key is a private stream you
hold as a value, and `jax.random.split` manufactures as many independent
streams as you need. The property a private stream is supposed to
protect, that unrelated draws elsewhere cannot shift yours, holds by
construction, because no draw advances any shared state. Here an
unrelated draw between two uses of the same key changes nothing:

In [ ]:
key_split, key_other = jax.random.split(jax.random.key(42))
split = jax.random.permutation(key_split, 10)
_ = jax.random.normal(key_other, (1000,))  # unrelated draw, its own key
split_again = jax.random.permutation(key_split, 10)
print(jnp.array_equal(split, split_again), split)

Every function in `jax.random` takes the key as its first argument; there
is no variant that consults hidden state.

### DataLoader Workers

Parallel loader workers still need separate random streams. With explicit
keys, those streams are visible in the program: a worker's randomness is
whatever key you hand it, so you split one key
into per-worker keys (`jax.random.split(key, num_workers)`), refresh them
each epoch by folding in the epoch number (`jax.random.fold_in`), and two
workers can share a stream only if the code visibly passes the same key
twice. JAX programs usually borrow their input
pipeline from NumPy, PyTorch, or `tf.data`, and those loaders bring their
own worker-seeding contract. When you do that, configure the loader's
workers from one base seed and refresh them each epoch.

### Randomness as a Value

Every failure above is hidden global state: a generator that lives
somewhere your code does not mention, silently copied or shared. Some
library designs remove the hiding place altogether. In an explicit-PRNG
design (JAX is the prominent example), randomness is threaded through the
program as a value: every random operation takes a key argument, using the
same key twice yields the same numbers by definition, and independent
streams are obtained by explicitly splitting a key in two. Accidentally
duplicating a stream would require writing the duplication into the code,
so the worker bug cannot be expressed. The price is bookkeeping, since
every function that needs randomness must accept and split keys. PyTorch's
global seed is the convenient version of the same contract, one implicit
key that everything shares; generator objects recover the explicit version
where it matters.

The two cells above are this design in action: `train_once` turned one
seed into one key and split it, and reusing a key reproduced the
permutation bit for bit.

## Determinism and Its Price

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic evaluates. Floating-point addition is not associative
(that section), so summing the same numbers in a different
order or grouping may give a different answer:

In [ ]:
x = jax.random.normal(jax.random.key(0), (1_000_000,))
s_fwd, s_rev = x.sum(), x[::-1].sum()  # same numbers, different order
print(s_fwd == s_rev, (s_fwd - s_rev).item())

For the *drawn numbers*, JAX's answer is structural. Its PRNG is
counter-based (Threefry): a draw is a pure function of the key and the
requested shape, not of any evolving state, so the same key produces the
same numbers on CPU, GPU, and TPU alike (the documented caveat is that
JAX does not promise identical bits *across its own releases*). For the
*arithmetic*, the story splits by backend. On CPU, XLA's kernels are
deterministic, and the bitwise agreement of `train_once` above is exactly
that, verified. On GPU, some XLA kernels commit partial sums via atomic
additions in whatever order threads finish, the same last-bits
nondeterminism as elsewhere; setting the environment variable
`XLA_FLAGS=--xla_gpu_deterministic_ops=true` before JAX initializes
forces deterministic implementations, at a speed cost we have not
measured here. There is no error-raising mode: the flag substitutes
deterministic kernels rather than refusing nondeterministic ones.

This guarantee has a narrow scope. No framework promises bitwise agreement
across releases, platforms, or CPU versus GPU: it holds only for a
pinned machine, library version, and flag configuration. That makes bitwise
reproducibility a *debugging* tool, the setting that lets you bisect
exactly where two runs diverge. The *scientific* goal is statistical
reproducibility: the same conclusions across seeds, reported as a mean and
spread over several runs rather than one fortunate curve. The distinction
mirrors that section: changing dtype changes results in the
last bits by design, and an experimental claim that survives neither a new
seed nor bfloat16 was never a result.

## Hooks: Looking Inside

NNX can wrap a module call with `nnx.capture`, recording the return value of
each submodule method alongside the model output without changing the model's
source. That matters
precisely when the model came from a library or a checkpoint you do not
want to edit. the figure draws the general picture: an
observation point in the gap the call wrapper already leaves around each
module's computation, from which an observer can capture or check
without a single line of the model changing.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

We reuse the residual stack of
that section, rebuilt compactly:

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

rngs = nnx.Rngs(0)
net = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                     *[ResidualBlock(64, rngs) for _ in range(8)],
                     nnx.Linear(64, 10, rngs=rngs))
X = jax.random.normal(jax.random.key(1), (256, 20))

### Capturing Activation Statistics

The initialization experiments of that section measured the
standard deviation of activations at every depth. The same measurement
takes a few lines on an unmodified model:

In [ ]:
_, inter = nnx.capture(
    net, nnx.Intermediate, method_outputs=nnx.Intermediate)(X)
for k, layer in enumerate(net.layers):
    out = inter['layers'][k]['__call__'][0]
    print(f'{type(layer).__name__:15s} std {out.std():.2f}')

The residual stream's spread grows block by block, since each block adds
its body's output on top of the stream, exactly the depth effect that
motivated the scaled initializations of that section, measured
here without touching the model.

Nothing here needs detaching or removing: the captured intermediates are
ordinary arrays with no autograd graph attached, the dictionary exists
for this one call, and no observer stays registered on the model. The
`method_outputs=nnx.Intermediate` argument records every method return. For
finer control, a module can call
`self.sow(nnx.Intermediate, 'name', value)` at selected points, and
`nnx.capture(model, nnx.Intermediate)` then returns only those named values.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. Check every leaf module's output for
finiteness and the answer arrives in one forward pass. We sabotage a
weight deep in the stack to watch it fire:

In [ ]:
saved = net.layers[3].body.layers[0].kernel[0, 0]
net.layers[3].body.layers[0].kernel[0, 0] = float('nan')
_, inter = nnx.capture(
    net, nnx.Intermediate, method_outputs=nnx.Intermediate)(X)

def get_path(tree, path):
    for key in path:
        tree = tree[key]
    return tree

for path, module in nnx.iter_modules(net):
    if isinstance(module, nnx.Linear):
        out = get_path(inter, path)['__call__'][0]
        if not jnp.isfinite(out).all():
            print('first non-finite output in', path)
            break

net.layers[3].body.layers[0].kernel[0, 0] = saved  # net is shared; undo the sabotage

We inspect the captured outputs of linear modules in object-graph order, which
matches execution order for this sequential network. The report names
`('layers', 3, 'body', 'layers', 0)`, the layer we poisoned, rather than
leaving you to bisect with print statements while NaNs propagate through
everything downstream.

### Backward Hooks and Beyond

Gradients need no hook at all, because they are not events that fire
inside a backward pass: `nnx.grad` returns them as a tree of values in
the same shape as the parameters. To log per-layer gradient norms, catch
exploding gradients at their source, or experiment with per-layer
clipping, compute the gradient tree and inspect or transform it like any
other data:

In [ ]:
grads = nnx.grad(lambda model: (model(X) ** 2).mean())(net)
norms = jax.tree_util.tree_map(jnp.linalg.norm, grads)
print(norms['layers'][3]['body']['layers'][0])

## Summary

Reproducibility is an inventory problem: initialization, dropout,
shuffling, augmentation, and loader workers each draw from some generator,
and a run repeats only if every one of them is seeded.

In JAX the inventory collapses to one item, the key you pass: the same
key gives the same draws by construction of the counter-based PRNG, and
independent streams come from `jax.random.split`, never from hidden
state. Keys make the program repeatable, not the arithmetic: on CPU,
XLA's kernels are already deterministic, while on GPU the flag
`--xla_gpu_deterministic_ops=true` pins kernel choice too.

Bitwise agreement is a debugging tool;
conclusions that hold across seeds are the scientific goal.

For looking inside a model, `nnx.capture(..., method_outputs=...)` returns
submodule outputs from an unmodified call, `sow` records named values from
the inside, and gradients are values from `nnx.grad` you inspect
directly.

## Exercises

1. Extend `train_once` to load data through your framework's parallel data
   loader (e.g., PyTorch's `DataLoader`, with `num_workers=4`). Give the
   dataset its own `np.random.default_rng()` object and use it to add noise
   as each example loads. On a process-based loader, check whether workers
   copied the same generator state. Using the per-worker seeding hook this
   section introduced for your framework, replace the dataset's stashed
   generator with one derived from the worker's own seed, then verify
   agreement across two runs.
2. Write a forward hook (or your framework's equivalent introduced in this
   section) that counts multiply-accumulate operations for every linear
   layer from the shapes of its input and weight, and use it to report
   per-layer and total FLOPs for the residual stack above. Check the total
   against a hand count.
3. Using a backward hook (or your framework's equivalent introduced in this
   section), record the norm of each block's output gradient. After the
   backward pass, compare these activation-gradient norms with the
   parameter-gradient norms in the same block. Which one first reveals an
   exploding backward signal?
4. A forward hook that returns a value *replaces* the module's output. Use
   one to zero out the output of a single residual block's body (turning
   the block into the identity) and measure how the network's output
   changes. Which block matters most, and how would you find out with one
   loop?

5. Rebuild the activation-statistics table by editing `ResidualBlock` to call
   `self.sow(nnx.Intermediate, 'body_out', ...)` on its body's output, then
   collect those values with `nnx.capture(net, nnx.Intermediate)`. Compare
   capture-all-methods, opt-in `sow`, and
   PyTorch-style mutable hooks: which requires touching model
   code, which can silently retain memory, and which would you want for a
   model you do not own?